# 02 — Data Assessment

> **Read-only audit of the raw dataset.**  
> No data is modified, imputed, or removed in this notebook.  
> Output: a structured quality report. Remediation actions are deferred to `03_data_cleaning.ipynb`.

## 1. Setup

In [ ]:
import hashlib
import logging
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

import itables.options as opt
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from itables import init_notebook_mode, show

from src.loader import load_config
# Load configuration settings from config.toml
config = load_config()

# Clean logging configuration tailored for Jupyter notebooks and scripts
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Initialize notebook mode for interactive display
init_notebook_mode(all_interactive=True)
opt.warn_on_undocumented_option = False

opt.maxBytes   = 0
opt.pageLength = 10
opt.lengthMenu = [10, 25, 50, 100]
opt.scrollX    = True
opt.classes    = "display nowrap compact"
opt.style      = "width:100%; font-size:13px;"

# Set seaborn theme and figure parameters
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120})

NOTEBOOK_VERSION = "1.0.3"
NULL_THRESHOLD   = 0.30   # critical completeness threshold
Z_THRESHOLD      = 3.0    # z-score outlier threshold
TOLERANCE        = 0.005  # rounding tolerance for pct cross-checks

log.info(f"Assessment notebook v{NOTEBOOK_VERSION} — initialized")

## 2. Load Raw Data

In [ ]:
# Retrieve settings from config
RAW_DATA_DIR = ROOT / config["data"]["raw_data_dir"]

start_year = config["settings"]["start_season"]
end_year   = config["settings"]["end_season"]

start_season = f"{start_year}-{str(start_year + 1)[-2:]}"
end_season   = f"{end_year}-{str(end_year + 1)[-2:]}"

# Define dataset version
DATASET_VERSION = f"{start_season}_{end_season}"

RAW_FILE = RAW_DATA_DIR / f"nba_stats_{DATASET_VERSION}.csv"

log.info(f"Source file     : {RAW_FILE}")
log.info(f"Dataset version : {DATASET_VERSION}")

sha256         = hashlib.sha256(RAW_FILE.read_bytes()).hexdigest()
load_timestamp = datetime.now().isoformat(timespec="seconds")

df = pl.read_csv(RAW_FILE, infer_schema_length=10_000, try_parse_dates=False)

n_rows, n_cols = df.shape

log.info(f"Loaded {n_rows:,} rows x {n_cols} columns")
log.info(f"SHA-256         : {sha256}")
log.info(f"Loaded at       : {load_timestamp}")

## 3. Structural Overview

In [ ]:
# Contractual schema: column → expected Polars dtype
EXPECTED_SCHEMA: dict[str, pl.DataType] = {
    # Identifiers (traditional split)
    "gameId":                          pl.String,
    "teamId":                          pl.String,
    "teamCity":                        pl.String,
    "teamName":                        pl.String,
    "teamTricode":                     pl.String,
    "teamSlug":                        pl.String,
    # Traditional stats (starter / bench split)
    "minutes":                         pl.String,
    "fieldGoalsMade":                  pl.Int64,
    "fieldGoalsAttempted":             pl.Int64,
    "fieldGoalsPercentage":            pl.Float64,
    "threePointersMade":               pl.Int64,
    "threePointersAttempted":          pl.Int64,
    "threePointersPercentage":         pl.Float64,
    "freeThrowsMade":                  pl.Int64,
    "freeThrowsAttempted":             pl.Int64,
    "freeThrowsPercentage":            pl.Float64,
    "reboundsOffensive":               pl.Int64,
    "reboundsDefensive":               pl.Int64,
    "reboundsTotal":                   pl.Int64,
    "assists":                         pl.Int64,
    "steals":                          pl.Int64,
    "blocks":                          pl.Int64,
    "turnovers":                       pl.Int64,
    "foulsPersonal":                   pl.Int64,
    "points":                          pl.Int64,
    "startersBench":                   pl.String,
    # Advanced stats (team-level aggregates, duplicated from join)
    "teamCity_adv":                    pl.String,
    "teamName_adv":                    pl.String,
    "teamTricode_adv":                 pl.String,
    "teamSlug_adv":                    pl.String,
    "minutes_adv":                     pl.String,
    "estimatedOffensiveRating":        pl.Float64,
    "offensiveRating":                 pl.Float64,
    "estimatedDefensiveRating":        pl.Float64,
    "defensiveRating":                 pl.Float64,
    "estimatedNetRating":              pl.Float64,
    "netRating":                       pl.Float64,
    "assistPercentage":                pl.Float64,
    "assistToTurnover":                pl.Float64,
    "assistRatio":                     pl.Float64,
    "offensiveReboundPercentage":      pl.Float64,
    "defensiveReboundPercentage":      pl.Float64,
    "reboundPercentage":               pl.Float64,
    "estimatedTeamTurnoverPercentage": pl.Float64,
    "turnoverRatio":                   pl.Float64,
    "effectiveFieldGoalPercentage":    pl.Float64,
    "trueShootingPercentage":          pl.Float64,
    "usagePercentage":                 pl.Float64,
    "estimatedUsagePercentage":        pl.Float64,
    "estimatedPace":                   pl.Float64,
    "pace":                            pl.Float64,
    "pacePer40":                       pl.Float64,
    "possessions":                     pl.Float64,
    "PIE":                             pl.Float64,
    # LeagueGameFinder columns (joined; partially redundant with above)
    "SEASON_ID":                       pl.String,
    "TEAM_ABBREVIATION":               pl.String,
    "TEAM_NAME":                       pl.String,
    "GAME_DATE":                       pl.String,
    "MATCHUP":                         pl.String,
    "WL":                              pl.String,
    "MIN":                             pl.Int64,
    "PTS":                             pl.Int64,
    "FGM":                             pl.Int64,
    "FGA":                             pl.Int64,
    "FG_PCT":                          pl.Float64,
    "FG3M":                            pl.Int64,
    "FG3A":                            pl.Int64,
    "FG3_PCT":                         pl.Float64,
    "FTM":                             pl.Int64,
    "FTA":                             pl.Int64,
    "FT_PCT":                          pl.Float64,
    "OREB":                            pl.Int64,
    "DREB":                            pl.Int64,
    "REB":                             pl.Int64,
    "AST":                             pl.Int64,
    "STL":                             pl.Int64,
    "BLK":                             pl.Int64,
    "TOV":                             pl.Int64,
    "PF":                              pl.Int64,
    "PLUS_MINUS":                      pl.Float64,
    "SEASON":                          pl.String,
}

actual_cols   = set(df.columns)
expected_cols = set(EXPECTED_SCHEMA.keys())

missing_cols    = expected_cols - actual_cols
unexpected_cols = actual_cols  - expected_cols

print(f"Shape            : {n_rows:,} rows x {n_cols} columns")
print(f"Expected columns : {len(expected_cols)}")
print(f"Actual columns   : {len(actual_cols)}")
print(f"Missing columns  : {missing_cols  or 'none'}")
print(f"Unexpected cols  : {unexpected_cols or 'none'}\n")

schema_rows = []

# Check schema consistency
for col in df.columns:
    actual_dtype   = str(df[col].dtype)
    expected_dtype = str(EXPECTED_SCHEMA.get(col, "NOT IN CONTRACT"))
    match_flag     = "OK" if actual_dtype == expected_dtype else "MISMATCH"
    schema_rows.append({
        "column":         col,
        "actual_dtype":   actual_dtype,
        "expected_dtype": expected_dtype,
        "match":          match_flag,
    })

# Create DataFrame of schema mismatches
df_schema = pl.DataFrame(schema_rows)
mismatches = df_schema.filter(pl.col("match") == "MISMATCH")
log.info(f"Schema mismatches: {mismatches.height}")

# Display schema mismatches
show(df_schema)

In [ ]:
# Retrieve assessment settings from config
preview_rows = config["assessment"]["preview_rows"]
sample_rows  = config["assessment"]["sample_rows"]
random_seed  = config["assessment"]["random_seed"]

# Display preview and sample rows
print("=== First 3 rows ===")
display(df.head(preview_rows))

print("\n=== Random 3-row sample (seed=42) ===")
display(df.sample(sample_rows, seed=random_seed))

## 4. Data Quality Assessment

### 4.1 Completeness

In [ ]:
# Calculate null counts for each column
null_counts = (
    df
    .null_count()
    .unpivot(variable_name="column", value_name="null_count")
    .with_columns(
        (pl.col("null_count") / n_rows).alias("null_pct")
    )
    .sort("null_pct", descending=True)
)

# Filter columns with nulls above threshold
critical_cols = null_counts.filter(pl.col("null_pct") > NULL_THRESHOLD)

# Count columns with any nulls
n_cols_any_null = null_counts.filter(pl.col("null_count") > 0).height
log.info(f"Columns with any nulls        : {n_cols_any_null}")

# Display critical columns
log.info(f"Critical columns (>{NULL_THRESHOLD:.0%} null)  : {critical_cols.height}")

if critical_cols.height > 0:
    log.warning(f"CRITICAL: {critical_cols['column'].to_list()}")

# Display columns with nulls only
print("\n--- Null counts (columns with nulls only) ---")
display(null_counts.filter(pl.col("null_count") > 0))

# Filter completely empty records
empty_records = df.filter(
    pl.all_horizontal([pl.col(c).is_null() for c in df.columns])
)
log.info(f"Completely empty records        : {empty_records.height}")

### 4.2 Uniqueness

In [ ]:
# Calculate exact duplicate rows
n_exact_dupes = df.height - df.unique().height
log.info(f"Exact duplicate rows: {n_exact_dupes:,}")

# Display sample of exact duplicates
if n_exact_dupes > 0:
    dupes = df.filter(df.is_duplicated())
    log.info(f"Sample of exact duplicates ({min(5, dupes.height)} of {dupes.height}):")
    display(dupes.head(5))
else:
    log.info("No exact duplicate rows found.")

In [ ]:
# Business key: (gameId, teamId, startersBench) — must be unique
BK_COLS = ["gameId", "teamId", "startersBench"]

# Calculate duplicate (gameId, teamId, startersBench) combos
bk_dupes = (
    df
    .group_by(BK_COLS)
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
    .sort("count", descending=True)
)
log.info(f"Duplicate (gameId, teamId, startersBench) combos: {bk_dupes.height:,}")

if bk_dupes.height > 0:
    print(bk_dupes.head(10))

# Each (gameId, teamId) pair must have exactly 2 rows: Starters + Bench
lgf_anomalies = (
    df
    .group_by(["gameId", "teamId"])
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") != 2)
    .sort("count", descending=True)
)
log.info(
    f"(gameId, teamId) combos with count != 2 "
    f"(expected: Starters + Bench): {lgf_anomalies.height:,}"
)
if lgf_anomalies.height > 0:
    print(lgf_anomalies.head(10))

In [ ]:
# Near-duplicates: same game + team but unexpected split count
near_dup_candidates = (
    df
    .group_by(["gameId", "teamId"])
    .agg(
        pl.col("startersBench").n_unique().alias("n_splits"),
        pl.len().alias("n_rows"),
    )
    .filter(pl.col("n_splits") > 2)
    .sort("n_rows", descending=True)
)
log.info(
    f"Games with more than 2 starter/bench splits "
    f"(potential near-duplicates): {near_dup_candidates.height:,}"
)
if near_dup_candidates.height > 0:
    display(near_dup_candidates.head(10))
else:
    log.info("No near-duplicate patterns detected.")

### 4.3 Validity

In [ ]:
# WL domain: {W, L}
wl_values = df["WL"].drop_nulls().unique().sort().to_list()
log.info(f"WL distinct values      : {wl_values}")
invalid_wl = df.filter(
    ~pl.col("WL").is_in(["W", "L"]) & pl.col("WL").is_not_null()
)
log.info(f"Invalid WL records      : {invalid_wl.height}")

# startersBench domain: {Starters, Bench}
sb_values = df["startersBench"].drop_nulls().unique().sort().to_list()
log.info(f"startersBench values    : {sb_values}")
# Select records with invalid startersBench values
invalid_sb = df.filter(
    ~pl.col("startersBench").is_in(["Starters", "Bench"])
    & pl.col("startersBench").is_not_null()
)
log.info(f"Invalid startersBench   : {invalid_sb.height}")

# MATCHUP: must contain " vs. " (home game) or " @ " (away game)
invalid_matchup = df.filter(
    ~pl.col("MATCHUP").str.contains(r" vs\. | @ ")
    & pl.col("MATCHUP").is_not_null()
)
log.info(f"Invalid MATCHUP format  : {invalid_matchup.height}")

# SEASON: must match YYYY-YY pattern
invalid_season = df.filter(
    ~pl.col("SEASON").str.contains(r"^\d{4}-\d{2}$")
    & pl.col("SEASON").is_not_null()
)
log.info(f"Invalid SEASON format   : {invalid_season.height}")

if invalid_wl.height > 0:
    print("Invalid WL sample:"); print(invalid_wl.head(5))
if invalid_sb.height > 0:
    print("Invalid startersBench sample:"); print(invalid_sb.head(5))

In [ ]:
# Columns that must be >= 0
NON_NEGATIVE_COLS = [
    "fieldGoalsMade", "fieldGoalsAttempted",
    "threePointersMade", "threePointersAttempted",
    "freeThrowsMade", "freeThrowsAttempted",
    "reboundsOffensive", "reboundsDefensive", "reboundsTotal",
    "assists", "steals", "blocks", "turnovers", "foulsPersonal", "points",
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "MIN",
]

range_issues: dict[str, int] = {}
for col in NON_NEGATIVE_COLS:
    if col not in df.columns:
        continue
    n_neg = df.filter(
        pl.col(col).cast(pl.Float64, strict=False) < 0
    ).height
    if n_neg > 0:
        range_issues[col] = n_neg
        log.warning(f"Column '{col}': {n_neg} negative value(s)")

if not range_issues:
    log.info("No negative values found in non-negative columns")
else:
    print("Range violations:", range_issues)

In [ ]:
# Percentage columns must be in [0, 1]
PCT_COLS = [
    ("fieldGoalsPercentage",         0.0, 1.0),
    ("threePointersPercentage",      0.0, 1.0),
    ("freeThrowsPercentage",         0.0, 1.0),
    ("FG_PCT",                       0.0, 1.0),
    ("FG3_PCT",                      0.0, 1.0),
    ("FT_PCT",                       0.0, 1.0),
    ("effectiveFieldGoalPercentage", 0.0, 1.0),
    ("trueShootingPercentage",       0.0, 1.0),
]

pct_issues: dict[str, int] = {}
for col, lo, hi in PCT_COLS:
    if col not in df.columns:
        continue
    n_bad = df.filter(
        pl.col(col).cast(pl.Float64, strict=False).is_not_null()
        & (
            (pl.col(col).cast(pl.Float64, strict=False) < lo)
            | (pl.col(col).cast(pl.Float64, strict=False) > hi)
        )
    ).height
    if n_bad > 0:
        pct_issues[col] = n_bad
        log.warning(f"Percentage column '{col}': {n_bad} value(s) outside [{lo}, {hi}]")

if not pct_issues:
    log.info("All percentage columns are within [0, 1]")
else:
    print("Percentage violations:", pct_issues)

In [ ]:
invalid_date_format = df.filter(
    ~pl.col("GAME_DATE").str.contains(r"^\d{4}-\d{2}-\d{2}$")
    & pl.col("GAME_DATE").is_not_null()
)
log.info(f"Records with invalid GAME_DATE format : {invalid_date_format.height}")

# Convert GAME_DATE to datetime and filter out invalid dates
df_dates = df.with_columns(
    pl.col("GAME_DATE")
    .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
    .alias("_date")
)
min_date = df_dates["_date"].min()
max_date = df_dates["_date"].max()
log.info(f"GAME_DATE observed range: {min_date} -> {max_date}")

# Get earliest and latest dates from config
earliest_date = config["assessment"]["earliest_date"]
latest_date = config["assessment"]["latest_date"]

# Convert config dates to datetime objects
earliest_date = datetime.strptime(earliest_date, "%Y-%m-%d").date()
latest_date = datetime.strptime(latest_date, "%Y-%m-%d").date()

# Filter records with GAME_DATE outside the specified range
out_of_range = df_dates.filter(
    (pl.col("_date") < earliest_date) | (pl.col("_date") > latest_date)
)
log.info(
    f"Records with GAME_DATE outside [{earliest_date}, {latest_date}]: {out_of_range.height}"
)
if out_of_range.height > 0:
    display(out_of_range.select(["gameId", "GAME_DATE", "SEASON"]).head(10))

### 4.4 Consistency

In [ ]:
# Cast column to float64 and return expression
def _cast(col: str) -> pl.Expr:
    return pl.col(col).cast(pl.Float64, strict=False)

# Dictionary to store consistency flags and counts
consistency_flags: dict[str, int] = {}

checks = [
    ("fieldGoalsMade > fieldGoalsAttempted",
     _cast("fieldGoalsMade") > _cast("fieldGoalsAttempted")),
    ("threePointersMade > threePointersAttempted",
     _cast("threePointersMade") > _cast("threePointersAttempted")),
    ("freeThrowsMade > freeThrowsAttempted",
     _cast("freeThrowsMade") > _cast("freeThrowsAttempted")),
    ("reboundsOff + reboundsDef != reboundsTotal",
     ((_cast("reboundsOffensive") + _cast("reboundsDefensive")) != _cast("reboundsTotal"))
     & pl.col("reboundsTotal").is_not_null()),
    ("FGM > FGA",  _cast("FGM")  > _cast("FGA")),
    ("FG3M > FG3A", _cast("FG3M") > _cast("FG3A")),
    ("FTM > FTA",  _cast("FTM")  > _cast("FTA")),
    ("OREB + DREB != REB",
     ((_cast("OREB") + _cast("DREB")) != _cast("REB")) & pl.col("REB").is_not_null()),
]

# Iterate through checks and count inconsistencies
for label, expr in checks:
    n = df.filter(expr).height
    consistency_flags[label] = n

print("\nConsistency cross-checks:")
for label, n in consistency_flags.items():
    status = "OK" if n == 0 else f"FAIL  ({n:,} records)"
    print(f"  {label:<55}  {status}")

In [ ]:
# Sanity check: verify stored percentages against recomputed values
pct_cross_checks = [
    ("FGM",  "FGA",  "FG_PCT"),
    ("FG3M", "FG3A", "FG3_PCT"),
    ("FTM",  "FTA",  "FT_PCT"),
]

n_mismatch: dict[str, int] = {}
for made, att, pct in pct_cross_checks:
    n = (
        df
        .filter(pl.col(att).cast(pl.Float64, strict=False) > 0)
        .with_columns(
            (
                pl.col(made).cast(pl.Float64, strict=False)
                / pl.col(att).cast(pl.Float64, strict=False)
            ).alias("_calc")
        )
        .filter(
            (pl.col("_calc") - pl.col(pct).cast(pl.Float64, strict=False)).abs()
            > TOLERANCE
        )
        .height
    )
    n_mismatch[pct] = n
    status = "OK" if n == 0 else f"MISMATCH  ({n:,} records)"
    log.info(f"{pct} recompute check: {status}")

print("\nStored vs. recomputed percentage checks (tolerance =", TOLERANCE, "):")
for pct, n in n_mismatch.items():
    print(f"  {pct:<12}  {'OK' if n == 0 else f'FAIL ({n:,})'}")

In [ ]:
# SEASON_ID encodes the season start year in digits 1-4 (NBA format: "2YYYY...")
# SEASON column: "YYYY-YY"
df_season_check = df.with_columns([
    # Cast to pl.String first so we can use string slicing
    pl.col("SEASON_ID").cast(pl.String).str.slice(1, 4).cast(pl.Int64, strict=False).alias("_sid_year"),
    pl.col("SEASON").cast(pl.String).str.slice(0, 4).cast(pl.Int64, strict=False).alias("_s_year"),
]).filter(
    pl.col("SEASON_ID").is_not_null() & pl.col("SEASON").is_not_null()
)

season_mismatch = df_season_check.filter(
    pl.col("_sid_year") != pl.col("_s_year")
)
log.info(f"SEASON_ID / SEASON year mismatch   : {season_mismatch.height:,} records")

if season_mismatch.height > 0:
    print(season_mismatch.select(["gameId", "SEASON_ID", "SEASON"]).head(10))
else:
    print("SEASON_ID and SEASON are consistent across all records.")

# gameId prefix convention: "002YYYXXXXX" where YYY = last 3 digits of start year
df_gid_check = df.with_columns([
    # Cast gameId and SEASON to String to avoid potential downstream dtype errors
    pl.col("gameId").cast(pl.String).str.slice(3, 3).cast(pl.Int64, strict=False).alias("_gid_year"),
    pl.col("SEASON").cast(pl.String).str.slice(2, 2).cast(pl.Int64, strict=False).alias("_season_2y"),
]).filter(pl.col("gameId").is_not_null() & pl.col("SEASON").is_not_null())

gid_mismatch = df_gid_check.filter(
    pl.col("_gid_year") != pl.col("_season_2y")
)
log.info(f"gameId year prefix / SEASON mismatch: {gid_mismatch.height:,} records")

if gid_mismatch.height > 0:
    display(gid_mismatch.select(["gameId", "SEASON"]).head(10))

### 4.5 Outlier Flagging (flag only — no removal)

In [ ]:
# Define numeric columns to check for outliers
NUMERIC_COLS = [
    "points", "assists", "reboundsTotal", "steals", "blocks", "turnovers",
    "fieldGoalsMade", "fieldGoalsAttempted",
    "threePointersMade", "threePointersAttempted",
    "freeThrowsMade", "freeThrowsAttempted",
    "offensiveRating", "defensiveRating", "netRating",
    "pace", "PIE",
    "PTS", "AST", "REB", "STL", "BLK", "TOV", "PLUS_MINUS",
]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]

outlier_summary: list[dict] = []

# Iterate through numeric columns and calculate outlier statistics
for col in NUMERIC_COLS:
    series = df[col].cast(pl.Float64, strict=False).drop_nulls()
    if series.len() == 0:
        continue

    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)
    iqr  = q3 - q1
    lo   = q1 - 1.5 * iqr
    hi   = q3 + 1.5 * iqr
    mean = series.mean()
    std  = series.std()

    # Calculate IQR outliers
    n_iqr = df.filter(
        (pl.col(col).cast(pl.Float64, strict=False) < lo)
        | (pl.col(col).cast(pl.Float64, strict=False) > hi)
    ).height

    # Calculate z-score outliers
    n_zscore = 0
    if std and std > 0:
        n_zscore = df.filter(
            (
                (pl.col(col).cast(pl.Float64, strict=False) - mean) / std
            ).abs() > Z_THRESHOLD
        ).height

    outlier_summary.append({
        "column":            col,
        "q1":                round(q1, 2),
        "q3":                round(q3, 2),
        "iqr_lower":         round(lo, 2),
        "iqr_upper":         round(hi, 2),
        "n_iqr_outliers":    n_iqr,
        "n_zscore_outliers": n_zscore,
    })

# Create DataFrame of outlier statistics
df_outlier_summary = pl.DataFrame(outlier_summary)
show(df_outlier_summary)

In [ ]:
PLOT_COLS = [
    "points", "assists", "reboundsTotal", "steals", "blocks",
    "offensiveRating", "defensiveRating", "pace", "PIE",
]
PLOT_COLS = [c for c in PLOT_COLS if c in df.columns]

n_plots = len(PLOT_COLS)
fig, axes = plt.subplots(n_plots, 2, figsize=(16, n_plots * 3))

for i, col in enumerate(PLOT_COLS):
    data = df[col].cast(pl.Float64, strict=False).drop_nulls().to_numpy()

    axes[i, 0].boxplot(
        data, 
        orientation='horizontal', 
        patch_artist=True,
        boxprops=dict(facecolor="#4c9be8", alpha=0.75),
    )
    
    axes[i, 0].set_title(f"{col}  —  boxplot", fontsize=10)
    axes[i, 0].set_xlabel(col)

    axes[i, 1].hist(data, bins=60, color="#4c9be8", edgecolor="white", alpha=0.85)
    axes[i, 1].set_title(f"{col}  —  histogram", fontsize=10)
    axes[i, 1].set_xlabel(col)
    axes[i, 1].set_ylabel("Count")

plt.suptitle("Distribution of Key Numeric Variables", fontsize=14, y=1.005)
plt.tight_layout()
plt.show()
log.info("Distribution plots rendered")

In [ ]:
# Flag records with z-score outliers
flagged_records: list[dict] = []

# Iterate through numeric columns and flag outliers
for col in NUMERIC_COLS:
    series = df[col].cast(pl.Float64, strict=False).drop_nulls()
    if series.len() == 0:
        continue
    mean = series.mean()
    std  = series.std()
    if not std or std == 0:
        continue

    flagged = (
        df
        .with_row_index("_row")
        .filter(
            (
                (pl.col(col).cast(pl.Float64, strict=False) - mean) / std
            ).abs() > Z_THRESHOLD
        )
        .select(["_row", "gameId", "teamId", "SEASON", col])
    )

    for row in flagged.iter_rows(named=True):
        val = row[col]
        z   = abs((float(val) - mean) / std) if val is not None else None
        flagged_records.append({
            "row":         row["_row"],
            "gameId":      row["gameId"],
            "teamId":      row["teamId"],
            "SEASON":      row["SEASON"],
            "flag_column": col,
            "value":       val,
            "z_score":     round(z, 2) if z is not None else None,
        })

log.info(f"Total flagged records (|z| > {Z_THRESHOLD}): {len(flagged_records):,}")

# Create DataFrame of flagged records
if flagged_records:
    df_flagged = pl.DataFrame(flagged_records)
    show(
        df_flagged
        .sort("z_score", descending=True)
        .head(20)
    )
else:
    df_flagged = pl.DataFrame()
    log.info("No records flagged as outliers.")

## 5. Assessment Report

In [ ]:
# Calculate total validity issues
total_validity_issues = (
    invalid_wl.height
    + invalid_sb.height
    + invalid_matchup.height
    + invalid_season.height
    + sum(range_issues.values())
    + sum(pct_issues.values())
    + invalid_date_format.height
    + out_of_range.height
)

# Calculate total consistency issues
total_consistency_issues = (
    sum(consistency_flags.values())
    + sum(n_mismatch.values())
    + season_mismatch.height
    + gid_mismatch.height
)

report_rows = [
    {
        "dimension":   "Completeness",
        "checks_run":  "null counts per column + empty records",
        "n_issues":    null_counts.filter(pl.col("null_count") > 0).height,
        "n_critical":  critical_cols.height,
        "status":      "FAIL" if critical_cols.height > 0 else "PASS",
    },
    {
        "dimension":   "Uniqueness",
        "checks_run":  "exact duplicates + business key + near-duplicates",
        "n_issues":    n_exact_dupes + bk_dupes.height,
        "n_critical":  n_exact_dupes,
        "status":      "FAIL" if n_exact_dupes > 0 else ("WARN" if bk_dupes.height > 0 else "PASS"),
    },
    {
        "dimension":   "Validity",
        "checks_run":  "domain + range + percentages + dates",
        "n_issues":    total_validity_issues,
        "n_critical":  invalid_wl.height + sum(range_issues.values()),
        "status":      "FAIL" if (invalid_wl.height + sum(range_issues.values())) > 0 else ("WARN" if total_validity_issues > 0 else "PASS"),
    },
    {
        "dimension":   "Consistency",
        "checks_run":  "arithmetic cross-checks + stored vs. computed + seasonal IDs",
        "n_issues":    total_consistency_issues,
        "n_critical":  sum(v for v in consistency_flags.values() if v > 0),
        "status":      "FAIL" if any(v > 0 for v in consistency_flags.values()) else ("WARN" if total_consistency_issues > 0 else "PASS"),
    },
    {
        "dimension":   "Outlier Flagging",
        "checks_run":  f"IQR + z-score (threshold={Z_THRESHOLD}) on {len(NUMERIC_COLS)} numeric columns",
        "n_issues":    len(flagged_records),
        "n_critical":  0,
        "status":      "INFO",
    },
]

df_report = pl.DataFrame(report_rows)
print("=== Assessment Report — Summary by Dimension ===\n")
display(df_report)

In [ ]:
# Calculate per-column completeness score
df_col_scores = (
    null_counts
    .with_columns(
        (1.0 - pl.col("null_pct")).round(4).alias("completeness_score")
    )
    .select(["column", "null_count", "null_pct", "completeness_score"])
)

print("=== Per-column Completeness Score ===\n")
show(df_col_scores)

In [ ]:
# Create list of issues with priority and impact
issues: list[dict] = []

if n_exact_dupes > 0:
    issues.append({"priority": 1, "impact": "HIGH",   "dimension": "Uniqueness",
                   "description": f"{n_exact_dupes:,} exact duplicate rows"})
if critical_cols.height > 0:
    issues.append({"priority": 2, "impact": "HIGH",   "dimension": "Completeness",
                   "description": f"{critical_cols.height} column(s) exceed {NULL_THRESHOLD:.0%} null: {critical_cols['column'].to_list()}"})
if any(v > 0 for v in consistency_flags.values()):
    bad = [k for k, v in consistency_flags.items() if v > 0]
    issues.append({"priority": 3, "impact": "HIGH",   "dimension": "Consistency",
                   "description": f"Arithmetic violations: {bad}"})
if bk_dupes.height > 0:
    issues.append({"priority": 4, "impact": "MEDIUM", "dimension": "Uniqueness",
                   "description": f"{bk_dupes.height:,} duplicate (gameId, teamId, startersBench) combos"})
if range_issues:
    issues.append({"priority": 5, "impact": "MEDIUM", "dimension": "Validity",
                   "description": f"Negative values in: {list(range_issues.keys())}"})
if pct_issues:
    issues.append({"priority": 6, "impact": "MEDIUM", "dimension": "Validity",
                   "description": f"Percentages outside [0,1] in: {list(pct_issues.keys())}"})
if any(v > 0 for v in n_mismatch.values()):
    bad_pcts = [k for k, v in n_mismatch.items() if v > 0]
    issues.append({"priority": 7, "impact": "MEDIUM", "dimension": "Consistency",
                   "description": f"Stored vs. computed PCT divergence (rounding?): {bad_pcts}"})
if season_mismatch.height > 0 or gid_mismatch.height > 0:
    issues.append({"priority": 8, "impact": "LOW",    "dimension": "Consistency",
                   "description": f"Seasonal ID mismatches: SEASON_ID/SEASON={season_mismatch.height}, gameId prefix={gid_mismatch.height}"})
if flagged_records:
    issues.append({"priority": 9, "impact": "LOW",    "dimension": "Outliers",
                   "description": f"{len(flagged_records):,} records flagged by |z| > {Z_THRESHOLD}"})

df_issues = (
    pl.DataFrame(issues)
    if issues
    else pl.DataFrame({"priority": [], "impact": [], "dimension": [], "description": []})
)

print("=== Prioritized Issue List ===\n")
display(df_issues.sort("priority"))

In [ ]:
deferred = [
    {
        "issue":       "Exact / business-key duplicates",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Removal strategy (keep-first vs. aggregate) depends on whether downstream "
            "features are computed at game-team or split level."
        ),
    },
    {
        "issue":       "Critical null columns",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Imputation vs. drop decision requires knowing which columns are features, "
            "labels, or purely informational."
        ),
    },
    {
        "issue":       "Arithmetic inconsistencies (made > attempted)",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Could be API artefacts or starter/bench aggregation edge cases. "
            "Requires per-game investigation before deciding to drop or recompute."
        ),
    },
    {
        "issue":       "Stored vs. computed PCT divergence",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Likely a rounding artefact from the NBA API. Decide whether to recompute "
            "percentages from raw counts or accept the stored values."
        ),
    },
    {
        "issue":       "Outlier records flagged by z-score",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Outliers are only flagged here; capping, Winsorization, or row removal "
            "requires domain validation and alignment with modelling goals."
        ),
    },
    {
        "issue":       "Redundant _adv suffix columns",
        "deferred_to": "03_data_cleaning.ipynb",
        "rationale":   (
            "Columns such as teamCity_adv, teamName_adv, etc. are join artefacts. "
            "Drop after confirming they are exact copies of their non-suffixed counterparts."
        ),
    },
]

df_deferred = pl.DataFrame(deferred)
print("=== Decisions Deferred to 03_data_cleaning.ipynb ===\n")
display(df_deferred)